# Temporal Convolutional Neural Network (TCN) for Turbofan Engine RUL Prediction

## Overview

This educational notebook demonstrates how to use a **Temporal Convolutional Neural Network (TCN)** for time series prediction of Remaining Useful Life (RUL) of turbofan engines.

## Learning Objectives

By the end of this notebook, you will:
- Understand how convolutional networks can process time series
- Learn about dilated convolutions for capturing long-range dependencies
- Build a TCN architecture with residual connections
- Compare TCN performance with RNN/LSTM approaches

## What is a Temporal CNN?

**Temporal Convolutional Networks** use 1D convolutions to process sequences, similar to how CNNs process images. TCNs are an alternative to RNNs for sequence modeling.

### Key Concepts:

1. **1D Convolutions**: Slide filters across the time dimension to detect patterns
2. **Dilated Convolutions**: Skip timesteps to capture long-range dependencies efficiently
3. **Residual Connections**: Allow gradients to flow through many layers (like ResNet for images)
4. **Causal Padding**: Ensures predictions only use past information (no future data leakage)

### Advantages:
- **Parallel processing**: Can process entire sequence at once (faster than RNNs)
- **Stable gradients**: No vanishing gradient problem
- **Flexible receptive field**: Dilated convolutions can see far into the past
- **Interpretable**: Can visualize what patterns the filters detect

### When to Use:
- When you need fast training and inference
- Long sequences where RNNs struggle
- When you want parallel processing
- When interpretability of learned patterns is important


## 1. Import Libraries

### Purpose
Import TensorFlow/Keras components for building TCN architecture, including Conv1D layers for temporal convolutions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv1D, Dense, Dropout, Input, Add, Activation, BatchNormalization, GlobalMaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')
sns.set_palette("husl")
%matplotlib inline

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")


## 2. Load and Prepare Data

### Purpose
Load the dataset. TCNs process sequences similarly to RNNs, but use convolution operations instead of recurrence.


In [ ]:
# Define data path
data_path = Path('../../dataset/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData')

# Column names with actual field names
op_settings = ['Altitude', 'Mach', 'TRA']
sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
           'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
column_names = ['unit', 'time'] + op_settings + sensors

def load_data(dataset='FD001'):
    """Load training and test data"""
    train_file = data_path / f'train_{dataset}.txt'
    test_file = data_path / f'test_{dataset}.txt'
    rul_file = data_path / f'RUL_{dataset}.txt'
    
    train_df = pd.read_csv(train_file, sep='\s+', header=None, names=column_names)
    test_df = pd.read_csv(test_file, sep='\s+', header=None, names=column_names)
    rul_df = pd.read_csv(rul_file, sep='\s+', header=None, names=['RUL'])
    
    return train_df, test_df, rul_df

def calculate_rul_train(df):
    """Calculate RUL for training data"""
    df = df.copy()
    df['RUL'] = df.groupby('unit')['time'].transform(lambda x: x.max() - x)
    return df

# Load data
train_df, test_df, rul_df = load_data('FD001')
train_df = calculate_rul_train(train_df)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Number of engines in training: {train_df['unit'].nunique()}")
print(f"Number of engines in test: {test_df['unit'].nunique()}")


## 3. Create Time Series Sequences

### Purpose
Create sequences for TCN processing. TCNs can process these sequences in parallel (unlike RNNs which process sequentially), making training faster.


In [ ]:
def create_sequences(data, sequence_length=30):
    """Create sequences for time series prediction"""
    sequences = []
    targets = []
    
    # Select features (using actual field names)
    op_settings = ['Altitude', 'Mach', 'TRA']
    sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
               'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
    feature_cols = op_settings + sensors
    
    for unit_id in data['unit'].unique():
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        unit_rul = unit_data['RUL'].values
        
        # Create sequences
        for i in range(len(unit_data) - sequence_length + 1):
            sequences.append(unit_features[i:i+sequence_length])
            targets.append(unit_rul[i+sequence_length-1])
    
    return np.array(sequences), np.array(targets)

# Create sequences
sequence_length = 30
X_train_seq, y_train_seq = create_sequences(train_df, sequence_length)

# For test data
def create_test_sequences(data, sequence_length=30):
    """Create test sequences"""
    sequences = []
    op_settings = ['Altitude', 'Mach', 'TRA']
    sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
               'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
    feature_cols = op_settings + sensors
    
    for unit_id in sorted(data['unit'].unique()):
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        
        if len(unit_features) >= sequence_length:
            sequences.append(unit_features[-sequence_length:])
        else:
            padding = np.zeros((sequence_length - len(unit_features), len(feature_cols)))
            sequences.append(np.vstack([padding, unit_features]))
    
    return np.array(sequences)

X_test_seq = create_test_sequences(test_df, sequence_length)
y_test = rul_df['RUL'].values

print(f"Training sequences shape: {X_train_seq.shape}")
print(f"Test sequences shape: {X_test_seq.shape}")


## 4. Data Normalization

### Purpose
Normalize data for TCN training. Convolutional layers are sensitive to input scale, so normalization is essential.


In [ ]:
# Normalize features
feature_scaler = MinMaxScaler()
n_samples, n_timesteps, n_features = X_train_seq.shape
X_train_reshaped = X_train_seq.reshape(-1, n_features)
X_train_scaled = feature_scaler.fit_transform(X_train_reshaped)
X_train_seq_scaled = X_train_scaled.reshape(n_samples, n_timesteps, n_features)

# Normalize test data
X_test_reshaped = X_test_seq.reshape(-1, n_features)
X_test_scaled = feature_scaler.transform(X_test_reshaped)
X_test_seq_scaled = X_test_scaled.reshape(X_test_seq.shape)

# Normalize targets
target_scaler = MinMaxScaler()
y_train_scaled = target_scaler.fit_transform(y_train_seq.reshape(-1, 1)).flatten()
y_test_scaled = target_scaler.transform(y_test.reshape(-1, 1)).flatten()

print("Data normalized successfully!")


## 5. Build Temporal CNN Model

### Purpose
Construct a TCN architecture with dilated convolutions and residual connections.

### Architecture Components:

1. **Temporal Blocks**: 
   - Two 1D convolutions with dilation
   - Batch normalization for stable training
   - Residual connection (adds input to output)
   - Helps with gradient flow and learning

2. **Dilated Convolutions**:
   - Dilation rate 1: Looks at adjacent timesteps
   - Dilation rate 2: Looks at every 2nd timestep (wider view)
   - Dilation rate 4: Looks at every 4th timestep (even wider)
   - Dilation rate 8: Looks at every 8th timestep (very wide view)
   - **Exponential growth**: Each block doubles the receptive field

3. **Residual Connections**:
   - Allow information to skip layers
   - Help with training deep networks
   - Enable learning of identity mappings

### Why This Architecture Works:
- **Multi-scale patterns**: Different dilation rates capture patterns at different time scales
- **Long-range dependencies**: High dilation rates can see far into the past
- **Stable training**: Residual connections prevent vanishing gradients


In [ ]:
def temporal_block(x, filters, kernel_size, dilation_rate):
    """Temporal block with dilated convolution and residual connection"""
    # Main path
    conv1 = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='same')(x)
    bn1 = BatchNormalization()(conv1)
    act1 = Activation('relu')(bn1)
    dropout1 = Dropout(0.2)(act1)
    
    conv2 = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='same')(dropout1)
    bn2 = BatchNormalization()(conv2)
    act2 = Activation('relu')(bn2)
    dropout2 = Dropout(0.2)(act2)
    
    # Residual connection
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    
    out = Add()([x, dropout2])
    return Activation('relu')(out)

def build_tcn_model(sequence_length, n_features):
    """Build Temporal CNN model"""
    inputs = Input(shape=(sequence_length, n_features))
    
    # Initial convolution
    x = Conv1D(64, 3, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # Temporal blocks with increasing dilation rates
    x = temporal_block(x, 64, 3, dilation_rate=1)
    x = temporal_block(x, 64, 3, dilation_rate=2)
    x = temporal_block(x, 64, 3, dilation_rate=4)
    x = temporal_block(x, 64, 3, dilation_rate=8)
    
    # Global pooling
    x = GlobalMaxPooling1D()(x)
    
    # Dense layers
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(1)(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Build model
n_features = X_train_seq_scaled.shape[2]
model = build_tcn_model(sequence_length, n_features)
model.summary()


## 6. Train Model

### Purpose
Train the TCN model. TCNs typically train faster than RNNs because:
- Convolutions can be parallelized
- No sequential dependency (unlike RNNs)
- Can process batches more efficiently


In [ ]:
# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Train model
history = model.fit(
    X_train_seq_scaled, y_train_scaled,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)


## 7. Evaluate Model

### Purpose
Evaluate TCN performance. Compare with RNN/LSTM to see if convolutional approach provides advantages.


In [ ]:
# Predictions
y_train_pred_scaled = model.predict(X_train_seq_scaled, verbose=0)
y_test_pred_scaled = model.predict(X_test_seq_scaled, verbose=0)

# Inverse transform
y_train_pred = target_scaler.inverse_transform(y_train_pred_scaled).flatten()
y_test_pred = target_scaler.inverse_transform(y_test_pred_scaled).flatten()

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train_seq, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train_seq, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_r2 = r2_score(y_train_seq, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*60)
print("Temporal CNN Model Performance")
print("="*60)
print(f"\nTraining Metrics:")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")
print(f"  R²:   {train_r2:.4f}")
print(f"\nTest Metrics:")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print(f"  R²:   {test_r2:.4f}")


## 8. Visualizations

### Purpose
Visualize TCN training and predictions. Compare training speed and final performance with RNN-based approaches.


In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Temporal CNN Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('Model MAE', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Predictions vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Temporal CNN Predictions vs Actual', fontsize=14, fontweight='bold')

# Training set
axes[0].scatter(y_train_seq, y_train_pred, alpha=0.5, s=20)
min_val = min(min(y_train_seq), min(y_train_pred))
max_val = max(max(y_train_seq), max(y_train_pred))
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual RUL', fontsize=11)
axes[0].set_ylabel('Predicted RUL', fontsize=11)
axes[0].set_title(f'Train Set (R² = {train_r2:.4f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, s=20)
min_val = min(min(y_test), min(y_test_pred))
max_val = max(max(y_test), max(y_test_pred))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual RUL', fontsize=11)
axes[1].set_ylabel('Predicted RUL', fontsize=11)
axes[1].set_title(f'Test Set (R² = {test_r2:.4f})', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
